In [1]:

# IMPORTS
import numpy as np
import pandas as pd
import requests


#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt
     

Question 1 What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?

In [7]:
from io import StringIO

In [8]:
def get_ipos_data() -> pd.DataFrame:
    
    url = f"https://www.iposcoop.com/ipos-recently-filed/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found.")

        return tables[0]

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [9]:
ipos = get_ipos_data()

In [10]:
ipos.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,Hacker Interstellar,SOUD,Kingswood Capital Markets,3.90,$10.50,$11.50,$42.90,TBA,S/O
4,2026-09-24,New Iceland Arctic Acquisition Corp.,NIAAU,Chardan,12.50,$10.00,$10.00,$125.00,Week of,S/O


In [16]:
ipos_withdrawl = ipos[
    (ipos['Expected To Trade'] == 'Withdrawn') 
    & (ipos['File Date'] <= '2026-09-11')
    ]
ipos_withdrawl.head()
len(ipos_withdrawl)

31

In [17]:
rules = {
    "Technologies": [r"\bTechnologies\b"],
    "Acquisition Corp": [
        r"\bAcquisition Corp\b",
        r"\bAcquisition Corporation\b",
        r"\bCorp\b"
    ],
    "Inc.": [
        r"\bInc\b",
        r"\bIncorporated\b"
    ],
    "Group": [r"\bGroup\b"],
    "Limited": [
        r"\bLtd\b",
        r"\bLimited\b"
    ],
    "Holdings": [
        r"\bHoldings\b",
        r"\bHolding\b"
    ]
}


In [18]:
ipos_withdrawl["Company Type"] = "Other"

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/2988296722.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Company Type"] = "Other"


In [21]:
for company_type, patterns in rules.items():
    pattern = "|".join(patterns)

    mask = (
        ipos_withdrawl["Company Type"].eq("Other") &
        ipos_withdrawl["Company"].str.contains(pattern, regex=True, na=False)
    )

    ipos_withdrawl.loc[mask, "Company Type"] = company_type

In [22]:
ipos_withdrawl["Company Type"].value_counts()

Company Type
Other               8
Limited             7
Acquisition Corp    5
Holdings            4
Technologies        3
Group               3
Inc.                1
Name: count, dtype: int64

In [23]:
def extract_numeric(value):
    return float(
        str(value)
        .replace("$", "")
        .replace(",", "")
        .strip()
    )

In [24]:
price_low = ipos_withdrawl["Price Low"].apply(extract_numeric)
price_high = ipos_withdrawl["Price High"].apply(extract_numeric)

ipos_withdrawl["Avg_price"] = (price_low + price_high) / 2

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/2081960412.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Avg_price"] = (price_low + price_high) / 2


In [25]:
ipos_withdrawl["Shares (millions)"] = pd.to_numeric(
    ipos_withdrawl["Shares (millions)"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False),
    errors="coerce"
)

ipos_withdrawl["Est $ Vol (millions)"] = pd.to_numeric(
    ipos_withdrawl["Est $ Vol (millions)"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
    errors="coerce"
)

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/764667097.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Shares (millions)"] = pd.to_numeric(
/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/764667097.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Est $ Vol (millions)"] = pd.to_numeric(


In [27]:
ipos_withdrawl["Shares_offered_value"] = (
    ipos_withdrawl["Shares (millions)"] * ipos_withdrawl["Avg_price"]
).fillna(ipos_withdrawl["Est $ Vol (millions)"])

/var/folders/f1/xm7l5z8n71x3s0xdf5072rfr0000gn/T/ipykernel_82850/2522762365.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ipos_withdrawl["Shares_offered_value"] = (


In [28]:
ipos_withdrawl.groupby("Company Type")["Shares_offered_value"].sum()

Company Type
Acquisition Corp    499.9850
Group                32.5000
Holdings            311.6575
Inc.                351.0000
Limited             197.2500
Other               290.4450
Technologies        184.9000
Name: Shares_offered_value, dtype: float64

<h2> Question 2

In [29]:
def get_ipos_data_q2() -> pd.DataFrame:
    
    url = f"https://www.iposcoop.com/2025-pricings/"
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Wrap HTML text in StringIO to avoid deprecation warning
        # "Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object."
        html_io = StringIO(response.text)
        tables = pd.read_html(html_io)

        if not tables:
            raise ValueError(f"No tables found.")

        return tables[0]

    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except ValueError as ve:
        print(f"Data error: {ve}")
    except Exception as ex:
        print(f"Unexpected error: {ex}")

    return pd.DataFrame()

In [32]:
df = get_ipos_data_q2()
len(df)

231

In [33]:
# Filtering: Filter the list to keep only those IPOs with an 'Offer Date' before 1 September 2025. 
# Also, exclude entries with a 0% return to ensure active tickers are processed. You should see 148 stocks.
# Convert Offer Date to datetime
df["Offer Date"] = pd.to_datetime(df["Offer Date"])

# Convert Return from strings such as "48.69%" to numbers
df["Return_numeric"] = pd.to_numeric(
    df["Return"].astype(str).str.replace("%", "", regex=False),
    errors="coerce"
)

# Apply filters
df = df[
    (df["Offer Date"] < pd.Timestamp("2025-09-01"))
    &
    (df["Return_numeric"] != 0)
].copy()

len(df)

146

In [34]:
import yfinance as yf

stocks = []

for ticker in df["Symbol"].dropna().unique():

    try:
        stock = yf.download(
            ticker,
            start="2025-01-01",
            end="2026-09-15",
            progress=False,
            auto_adjust=False
        )

        # Skip tickers not found in Yahoo Finance
        if stock.empty:
            print(f"{ticker} not found")
            continue

        # yfinance may return MultiIndex columns
        if isinstance(stock.columns, pd.MultiIndex):
            stock.columns = stock.columns.get_level_values(0)

        stock = stock.reset_index()

        # Keep track of which ticker each row belongs to
        stock["Ticker"] = ticker

        stocks.append(stock)

    except Exception as e:
        print(f"Error with {ticker}: {e}")


1 Failed download:
['MJID']: HTTPError('HTTP Error 404: ')


MJID not found



1 Failed download:
['EMPG']: HTTPError('HTTP Error 404: ')


EMPG not found



1 Failed download:
['CAEP']: YFTzMissingError('possibly delisted; no timezone found')


...

MTSR not found



1 Failed download:
['SKBL']: YFTzMissingError('possibly delisted; no timezone found')


SKBL not found



1 Failed download:
['MCTR']: YFTzMissingError('possibly delisted; no timezone found')


MCTR not found


In [35]:
stocks_df = pd.concat(stocks, ignore_index=True)

In [36]:
stocks_df["Ticker"].nunique()

132

In [42]:
stocks_df["growth_252d"] = (
    stocks_df.groupby("Ticker")["Close"]
    .transform(lambda x: x / x.shift(252))
)


In [46]:
# 30-day rolling volatility, separately for every ticker
stocks_df["volatility"] = (
    stocks_df.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(30).std())
    * np.sqrt(252)
)

In [47]:
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

In [48]:
snapshot = stocks_df[
    stocks_df["Date"] == pd.Timestamp("2026-09-11")
]

snapshot["Sharpe"].median()

0.049016452355376945

In [41]:
snapshot.describe()

/Users/cocochen/anaconda3/envs/tensorflow/lib/python3.11/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Price,Date,Adj Close,Close,High,Low,Open,Volume,growth_252d,volatility,Sharpe
count,132,132.000000,132.000000,132.000000,132.000000,132.000000,1.320000e+02,130.000000,132.000000,130.000000
mean,2026-09-11 00:00:00,12.800431,12.804076,13.185052,12.514412,12.835532,1.248240e+06,0.052117,28.205752,inf
min,2026-09-11 00:00:00,0.125000,0.125000,0.129000,0.115000,0.116000,0.000000e+00,-0.998995,0.000000,-2.534214
25%,2026-09-11 00:00:00,2.098750,2.098750,2.257500,2.075500,2.137500,2.297500e+04,-0.858965,2.338723,-0.171847
50%,2026-09-11 00:00:00,4.565000,4.565000,4.790000,4.475000,4.505000,1.708500e+05,-0.405466,8.369541,-0.037267
75%,2026-09-11 00:00:00,17.224999,17.224999,17.471250,16.732501,17.397501,9.976250e+05,0.038821,24.096908,-0.000343
max,2026-09-11 00:00:00,120.425003,120.425003,120.500000,120.425003,120.500000,2.168250e+07,32.638270,627.698106,inf
std,NaN,19.135136,19.138778,19.567789,18.872583,19.217228,3.184362e+06,3.049217,66.920670,NaN
